In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(
    str(Path("..").resolve())
)

from src.data_processing import (
    validate_schema,
    separate_valid_invalid_rows,
    prepare_data,
    get_new_records_by_occurrence,
    update_master_data,
    save_refreshed_master
)

from src.data_source import (
    load_recent_hdb_data_from_api
)



In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(
    str(Path("..").resolve())
)

from src.data_processing import (
    validate_schema,
    separate_valid_invalid_rows,
    prepare_data,
    get_new_records_by_occurrence
)

from src.data_source import (
    load_recent_hdb_data_from_api
)


In [3]:
MASTER_DATA_PATH = Path(
    "../data/processed/hdb_resale_clean.csv"
)

df_master = pd.read_csv(
    MASTER_DATA_PATH,
    parse_dates=["month"]
)

print("Master rows:", len(df_master))
print(
    "Master date range:",
    df_master["month"].min(),
    "to",
    df_master["month"].max()
)


Master rows: 244230
Master date range: 2017-01-01 00:00:00 to 2026-11-01 00:00:00


In [4]:
test_record = {
    "month": "2026-10",
    "town": "ANG MO KIO",
    "flat_type": "4 ROOM",
    "block": "123",
    "street_name": "ANG MO KIO AVE 3",
    "storey_range": "04 TO 06",
    "floor_area_sqm": 90.0,
    "flat_model": "Model A",
    "lease_commence_date": 1995,
    "remaining_lease": "67 years",
    "resale_price": 500000.0
}

test_master = pd.DataFrame([
    test_record,
    test_record
])

test_refresh = pd.DataFrame([
    test_record,
    test_record,
    test_record
])

test_new = get_new_records_by_occurrence(
    test_master,
    test_refresh
)

print("Master rows:", len(test_master))
print("Refresh rows:", len(test_refresh))
print("New rows found:", len(test_new))

assert len(test_new) == 1


Master rows: 2
Refresh rows: 3
New rows found: 1


In [5]:
latest_master_month = df_master["month"].max()

refresh_start_month = (
    latest_master_month
    - pd.DateOffset(months=2)
).strftime("%Y-%m")

print("Refresh from:", refresh_start_month)


Refresh from: 2026-09


In [6]:
refresh_end_month = (
    latest_master_month
    .strftime("%Y-%m")
)

print(
    "Refresh period:",
    refresh_start_month,
    "to",
    refresh_end_month
)

Refresh period: 2026-09 to 2026-11


In [7]:
df_api_raw = (
    load_recent_hdb_data_from_api(
        start_month=refresh_start_month,
        end_month=refresh_end_month,
        limit=1000
    )
)

print(
    "API rows:",
    len(df_api_raw)
)

2026-09: 2,239 rows
Rate limited. Retrying in 1s...
Rate limited. Retrying in 2s...
Rate limited. Retrying in 4s...
2026-10: 2,174 rows
2026-11: 443 rows
API rows: 4856


In [8]:
validate_schema(df_api_raw)

df_api_valid, df_api_quarantine = (
    separate_valid_invalid_rows(
        df_api_raw
    )
)

print("Valid API rows:", len(df_api_valid))
print("Quarantined API rows:", len(df_api_quarantine))


Valid API rows: 4856
Quarantined API rows: 0


In [9]:
df_api_clean = prepare_data(
    df_api_valid
)

print("Prepared API rows:", len(df_api_clean))


Prepared API rows: 4856


In [10]:
dtype_columns = [
    "floor_area_sqm",
    "lease_commence_date",
    "resale_price"
]

print("MASTER")
print(df_master[dtype_columns].dtypes)

print("\nAPI")
print(df_api_clean[dtype_columns].dtypes)


MASTER
floor_area_sqm         float64
lease_commence_date      int64
resale_price           float64
dtype: object

API
floor_area_sqm         float64
lease_commence_date      int64
resale_price           float64
dtype: object


In [11]:
df_master_updated, df_new_confirmed = (
    update_master_data(
        df_master,
        df_api_clean
    )
)

print(
    "Original master rows:",
    len(df_master)
)

print(
    "New rows:",
    len(df_new_confirmed)
)

print(
    "Updated master rows:",
    len(df_master_updated)
)

Original master rows: 244230
New rows: 85
Updated master rows: 244315


In [12]:
test_new_record = df_api_clean.iloc[[0]].copy()

test_new_record["block"] = "TEST999"

df_master_test, df_new_test = (
    update_master_data(
        df_master,
        pd.concat(
            [
                df_api_clean,
                test_new_record
            ],
            ignore_index=True
        )
    )
)

print(
    "Original master rows:",
    len(df_master)
)

print(
    "New rows detected:",
    len(df_new_test)
)

print(
    "Updated master rows:",
    len(df_master_test)
)

Original master rows: 244230
New rows detected: 86
Updated master rows: 244316


In [13]:
df_new = get_new_records_by_occurrence(
    df_master,
    df_api_clean
)

print("New records found:", len(df_new))


New records found: 85


In [14]:
display_columns = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "remaining_lease",
    "resale_price"
]

df_new[display_columns].head(20)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,remaining_lease,resale_price
4413,2026-11-01,ANG MO KIO,3 ROOM,208,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,48 years 09 months,418800.0
4415,2026-11-01,ANG MO KIO,3 ROOM,503,ANG MO KIO AVE 5,04 TO 06,68.0,New Generation,53 years 03 months,450000.0
4417,2026-11-01,ANG MO KIO,4 ROOM,228B,ANG MO KIO ST 23,01 TO 03,93.0,Model A,93 years 11 months,905000.0
4418,2026-11-01,ANG MO KIO,4 ROOM,231,ANG MO KIO AVE 3,07 TO 09,92.0,New Generation,49 years 11 months,560000.0
4420,2026-11-01,ANG MO KIO,4 ROOM,415,ANG MO KIO AVE 10,10 TO 12,92.0,New Generation,51 years 10 months,568000.0
4427,2026-11-01,BEDOK,3 ROOM,1,BEDOK STH AVE 1,13 TO 15,59.0,Improved,48 years 03 months,390888.0
4428,2026-11-01,BEDOK,3 ROOM,117,BEDOK NTH RD,10 TO 12,73.0,New Generation,50 years 07 months,430000.0
4437,2026-11-01,BEDOK,4 ROOM,221B,BEDOK CTRL,01 TO 03,94.0,Model A,82 years 07 months,888888.0
4443,2026-11-01,BEDOK,4 ROOM,808B,CHAI CHEE RD,13 TO 15,93.0,Model A,89 years 01 month,835000.0
4459,2026-11-01,BUKIT BATOK,2 ROOM,292A,BT BATOK EAST AVE 6,25 TO 27,47.0,Model A,91 years 03 months,405000.0


In [15]:
TEST_OUTPUT_DIR = Path(
    "../data/refresh_test_output"
)

test_master_path, test_snapshot_path = (
    save_refreshed_master(
        master_df=df_master_test,
        base_dir=TEST_OUTPUT_DIR,
        snapshot_date="2026-11-06"
    )
)

print(
    "Master saved to:",
    test_master_path
)

print(
    "Snapshot saved to:",
    test_snapshot_path
)

Master saved to: ..\data\refresh_test_output\master\hdb_resale_clean.csv
Snapshot saved to: ..\data\refresh_test_output\snapshots\2026-11-06\hdb_resale_clean.csv


In [16]:
print(
    "Master exists:",
    test_master_path.exists()
)

print(
    "Snapshot exists:",
    test_snapshot_path.exists()
)

Master exists: True
Snapshot exists: True


In [17]:
df_test_master_saved = pd.read_csv(
    test_master_path
)

df_test_snapshot_saved = pd.read_csv(
    test_snapshot_path
)

print(
    "Saved master rows:",
    len(df_test_master_saved)
)

print(
    "Saved snapshot rows:",
    len(df_test_snapshot_saved)
)

print(
    "Expected rows:",
    len(df_master_test)
)

Saved master rows: 244316
Saved snapshot rows: 244316
Expected rows: 244316


In [18]:
assert len(df_test_master_saved) == len(df_master_test)

assert len(df_test_snapshot_saved) == len(df_master_test)

print("Save verification passed.")

Save verification passed.
